# DS-3002 <br> Data Mining - Assignment 2

**Muhammad Moiz Khalid** <br>
**23i-2552**<br>
**BDS-6C**

## Preprocessing

In [ ]:
import re, json, random, time, os
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

os.makedirs("embeddings", exist_ok=True)
print("Imports OK | PyTorch:", torch.__version__)


## Data Loading & Tokenisation

In [ ]:
def load_articles(filepath):
    """Parse numbered [N] articles from cleaned.txt / raw.txt."""
    with open(filepath, encoding='utf-8') as f:
        content = f.read()
    parts = re.split(r'\[(\d+)\]', content)
    articles = {}
    for i in range(1, len(parts), 2):
        text = parts[i + 1].strip()
        if text:
            articles[int(parts[i])] = text
    return articles

def tokenize(text):
    """Keep only Urdu Unicode characters; split on whitespace."""
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    return [t for t in text.split() if t]

cleaned_articles = load_articles('cleaned.txt')
raw_articles = load_articles('raw.txt')
metadata = json.load(open('Metadata.json', encoding='utf-8'))

tokenized_cleaned = {aid: tokenize(t) for aid, t in cleaned_articles.items()}
tokenized_raw = {aid: tokenize(t) for aid, t in raw_articles.items()}

doc_ids = sorted(cleaned_articles.keys())

print(f"Cleaned articles : {len(cleaned_articles)}")
print(f"Raw articles : {len(raw_articles)}")
print(f"Metadata entries : {len(metadata)}")

### Topic Assignment (keyword-based, 5 categories)

In [ ]:
TOPIC_KEYWORDS = {
    'Politics':       ['الیکشن', 'حکومت', 'وزیر', 'پارلیمنٹ', 'سیاست',
                       'جماعت', 'ووٹ', 'وزیراعظم', 'صدر', 'اسمبلی'],
    'Sports':         ['کرکٹ', 'میچ', 'ٹیم', 'کھلاڑی', 'ورلڈکپ',
                       'فٹبال', 'کھیل', 'ٹورنامنٹ', 'وکٹ', 'گول'],
    'Economy':        ['مہنگائی', 'تجارت', 'بینک', 'بجٹ', 'معیشت',
                       'روپیہ', 'قرض', 'سرمایہ', 'ٹیکس', 'ڈالر'],
    'International':  ['اقوام', 'معاہدہ', 'امریکہ', 'چین', 'ایران',
                       'یوکرین', 'اسرائیل', 'روس', 'غیرملکی', 'تنازع'],
    'Health_Society': ['ہسپتال', 'بیماری', 'ویکسین', 'سیلاب', 'تعلیم',
                       'صحت', 'وبا', 'ڈاکٹر', 'آفت', 'امداد'],
}

def assign_topic(article_id):
    combined = (metadata.get(str(article_id), {}).get('title', '') +
                cleaned_articles.get(article_id, ''))
    scores = {t: sum(combined.count(k) for k in kws)
              for t, kws in TOPIC_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'International'

article_topics = {aid: assign_topic(aid) for aid in cleaned_articles}

print("Topic distribution:")
for t, c in sorted(Counter(article_topics.values()).items()):
    print(f"  {t:<20}: {c}")
